## Self-Hosted Agent Memory with Lakebase

### Installing Utilities and Libraries

In [ ]:
%pip install psycopg[binary]==3.3.4 psycopg_pool==3.3.1 "databricks-sdk>=0.89.0" langchain-community==0.4.1 databricks-openai==0.17.1

### Restart the Python Environment

In [ ]:
dbutils.library.restartPython()

### Setting up the Environment

In [ ]:
from databricks.sdk import WorkspaceClient
import psycopg

# Databricks SDK uses your existing OAuth identity
w = WorkspaceClient()

# Lakebase endpoint resource name
endpoint = (
    "projects/<project-id>/"
    "branches/<branch-id>/"
    "endpoints/<endpoint-id>"
)

# Generate a short-lived OAuth credential
credential = w.postgres.generate_database_credential(
    endpoint=endpoint
)

In [ ]:
host = "LAKEBASE_HOTSNAME"
db_name = "LAKEBASE_DATABASE_NAME"
username = "LAKEBASE_USERNAME"
password = credential.token

### Create a Connection Pool

In [ ]:
from psycopg_pool import ConnectionPool

pool = ConnectionPool(
    conninfo=(
        f"host={host} "
        f"dbname={db_name} "
        f"user={username} "
        f"password={password} "
        f"sslmode=require"
    ),
    min_size=2,
    max_size=10
)

pool.wait()

print("Connection pool created successfully")

### Create the Agent Memory Table

In [ ]:
create_table_query = """
CREATE SCHEMA IF NOT EXISTS agent_memory;

CREATE TABLE IF NOT EXISTS agent_memory.conversation_messages (

    message_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    thread_id TEXT NOT NULL,

    role TEXT NOT NULL,

    content TEXT NOT NULL,

    created_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
);
"""

with pool.connection() as conn:
    with conn.cursor() as cur:
        cur.execute(create_table_query)

    conn.commit()

print("Agent memory table created.")

### Create an Index for Memory Lookup

In [ ]:
create_index_query = """
CREATE INDEX IF NOT EXISTS idx_conversation_thread
ON agent_memory.conversation_messages(thread_id);
"""

with pool.connection() as conn:
    with conn.cursor() as cur:
        cur.execute(create_index_query)

    conn.commit()

print("Memory index created.")

### Helper Function to save a Memory Snippet

In [ ]:
def save_message(thread_id, role, content):

    query = """
    INSERT INTO agent_memory.conversation_messages
        (thread_id, role, content)
    VALUES
        (%s, %s, %s)
    """

    with pool.connection() as conn:
        with conn.cursor() as cur:
            cur.execute(
                query,
                (thread_id, role, content)
            )

        conn.commit()

### Helper Function to Retrieve Conversation Memory

In [ ]:
def get_conversation_memory(thread_id, limit=10):

    query = """
    SELECT role, content
    FROM agent_memory.conversation_messages
    WHERE thread_id = %s
    ORDER BY created_at DESC
    LIMIT %s
    """

    with pool.connection() as conn:
        with conn.cursor() as cur:

            cur.execute(
                query,
                (thread_id, limit)
            )

            rows = cur.fetchall()

    # Reverse because SQL returned newest first
    rows.reverse()

    return [
        {
            "role": role,
            "content": content
        }
        for role, content in rows
    ]

### Create the Databricks OpenAI Client

In [ ]:
from databricks_openai import DatabricksOpenAI

client = DatabricksOpenAI()

### Build the Memory Enabled Agent 

In [ ]:
def run_agent(thread_id, user_query):

    # Retrieve previous memory
    memory = get_conversation_memory(
        thread_id,
        limit=10
    )

    # Build model messages
    messages = [
        {
            "role": "system",
            "content": """
You are an ESG assistant.

Use previous conversation context when answering.
If the user refers to something discussed earlier,
use the stored memory to understand the reference.
"""
        }
    ]

    messages.extend(memory)

    messages.append(
        {
            "role": "user",
            "content": user_query
        }
    )

    # Call the model
    response = client.chat.completions.create(
        model="databricks-claude-sonnet-4-5",
        messages=messages
    )

    answer = response.choices[0].message.content

    # save the conversation state
    save_message(
        thread_id,
        "user",
        user_query
    )

    save_message(
        thread_id,
        "assistant",
        answer
    )


    return answer

### Test the Self-Hosted Memory Store

In [ ]:
thread_id = "esg-demo-001"

response = run_agent(
    thread_id,
    "My company is CarbonOps and we help companies prepare ESG reports."
)

print(response)

In [ ]:
response = run_agent(
    thread_id,
    "What does my company do?"
)

print(response)

### Retrieve the stored Memory Snippets

In [ ]:
from psycopg.rows import dict_row

query = """
SELECT
    thread_id,
    role,
    content,
    created_at
FROM agent_memory.conversation_messages
ORDER BY created_at;
"""

with pool.connection() as conn:

    with conn.cursor(row_factory=dict_row) as cur:

        cur.execute(query)

        rows = cur.fetchall()

for row in rows:
    print(row)

### Demonstrate Thread Isolation

In [ ]:
response = run_agent(
    "esg-demo-002",
    "My company builds electric vehicles."
)

print(response)

In [ ]:
response = run_agent(
    "esg-demo-002",
    "What does my company do?"
)

print(response)